# LLM-Based Psycholinguistic Feature Extraction

This notebook documents the prompt templates and extraction pipeline used in:  
**"A Large Language Model as a Psychometric Instrument for Occupational Stress Monitoring: A 15-Day Ecological Momentary Assessment Study"**

## Extraction Configuration

| Parameter | Value |
|---|---|
| Model | `gpt-5.2-2025-12-11` |
| API | OpenAI Chat Completions |
| Temperature | 0.0 |
| Output format | Structured JSON (`response_format`) |
| Extraction date | December 15, 2025 |
| Retry policy | Up to 2 retries on malformed JSON |
| Rate limit | 1.0 s fixed delay between requests |

## 13 Psycholinguistic Constructs

| Output Key | Construct |
|---|---|
| `posemo` | Positive emotion |
| `negemo` | Negative emotion |
| `anxiety` | Anxiety |
| `anger` | Anger |
| `sadness` | Sadness |
| `cogproc` | Cognitive processing |
| `tentativeness` | Tentativeness |
| `self_focus` | Self-focus |
| `social` | Social references |
| `work` | Work-related content |
| `time_pressure` | Time pressure |
| `somatic` | Somatic complaints |
| `coping` | Coping |

## Note on Korean Text in Prompts

The Korean-language lexical examples in the prompts below are integral to the extraction methodology.  
They must be preserved for reproducibility, as the model was instructed to score Korean daily diary texts.

---
## 1. System Prompt

In [ ]:
SYSTEM_PROMPT = """You are a PhD-level psychologist specializing in psycholinguistic analysis.
Your role is to extract quantitative, LIWC-informed linguistic features
from Korean daily diary texts written by working adults.

## PURPOSE
This task is part of an academic research study examining relationships
between language use and self-reported stress.
You are NOT asked to evaluate stress, mental health, or well-being.
You ONLY quantify language features.

## CORE PRINCIPLES
1. Base all scores STRICTLY on explicit linguistic content.
2. Do NOT infer psychological states beyond what is directly expressed.
3. Do NOT normalize or rescale scores based on perceived stress.
4. Be conservative, consistent, and reproducible.
5. When uncertain, prefer lower rather than higher scores.

## SCORING FRAMEWORK
- Scores represent the relative proportion of language devoted to each category.
- All values must be continuous between 0.0 and 1.0.
- Categories are NOT mutually exclusive.
- negemo (general) and anxiety/anger/sadness (specific) are scored independently.

## SCALE ANCHORS
- 0.0: Absent
- 0.1-0.2: Minimal
- 0.3-0.4: Moderate
- 0.5-0.6: Notable
- 0.7-0.8: Strong
- 0.9-1.0: Dominant

## REQUIRED OUTPUT KEYS
posemo, negemo, anxiety, anger, sadness,
cogproc, tentativeness,
self_focus, social,
work, time_pressure,
somatic, coping

## OUTPUT CONSTRAINTS
- Output MUST be a single valid JSON object.
- No explanations or text outside JSON.
- All values must be numeric floats.
"""

---
## 2. User Prompt Template

In [ ]:
USER_PROMPT_TEMPLATE = '''Analyze linguistic features in the following Korean daily diary text.
The text may include multiple time points within the same day.
Focus only on what is explicitly written.

## IMPORTANT: KOREAN LINGUISTIC FEATURES
Korean is a pro-drop language where subjects (especially "I") are frequently omitted.
For self_focus, count BOTH explicit pronouns AND implied first-person references.

## CATEGORY DEFINITIONS (LIWC-informed, Korean-adapted)

### AFFECTIVE LANGUAGE
- posemo:
  Positive emotion expressions.
  Examples: 좋다, 감사, 뿌듯, 기쁘다, 다행, 행복, 만족, 괜찮다
- negemo:
  General negative emotion expressions (broad category).
  Examples: 힘들다, 스트레스, 나쁘다, 싫다, 안 좋다
- anxiety:
  Worry, fear, or uncertainty about future outcomes.
  Examples: 걱정, 불안, 두렵다, 초조, 어떻게 될지 모르겠다
- anger:
  Frustration, irritation, or anger.
  Examples: 짜증, 화, 열받다, 답답, 빡치다
- sadness:
  Low mood, discouragement, or helplessness.
  Examples: 우울, 슬프다, 속상, 서럽다, 울적

### COGNITIVE LANGUAGE
- cogproc:
  Thinking, reasoning, or causal processing.
  Examples: 왜냐하면, 때문에, 그래서, ~다 보니까, 생각해보면
- tentativeness:
  Uncertainty, hedging, or lack of confidence.
  Examples: 것 같다, 아마, 모르겠다, 글쎄
  (Score genuine uncertainty, not just polite softening.)

### SELF & SOCIAL FOCUS
- self_focus:
  First-person singular self-reference.
  In Korean, the subject "I" is often omitted (pro-drop).
  Count BOTH:
    - Explicit: 나, 저, 내가, 제가, 나는, 저는
    - Implied: Sentences where "I" is the understood subject
      (e.g., "피곤했다" = "나는 피곤했다", "출근했다" = "내가 출근했다")
- social:
  Mentions of other people or interpersonal interactions.
  Examples: 동료, 친구, 가족, 고객, 민원인, 사람들

### CONTEXTUAL LANGUAGE
- work:
  Job- or work-related content.
  Examples: 업무, 일, 회의, 민원, 통화, 근무, 출근, 퇴근
- time_pressure:
  Urgency or time constraints.
  Examples: 바쁘다, 시간 없다, 오래, 길었다, 계속

### PHYSICAL & COPING LANGUAGE
- somatic:
  Physical states, fatigue, or bodily conditions.
  Examples: 피곤, 지치다, 컨디션, 기운, 잠, 몸
- coping:
  Coping, endurance, or self-encouragement.
  Examples: 버티다, 견디다, 잘했다, 수고했다, 마무리했다

---
## TEXT TO ANALYZE
"""
{text}
"""

## OUTPUT (JSON ONLY)
'''

---
## 3. Extraction Function

In [ ]:
from openai import OpenAI
import json
import time

client = OpenAI(api_key="YOUR_API_KEY")  # replace with your key

FEATURE_KEYS = [
    "posemo", "negemo", "anxiety", "anger", "sadness",
    "cogproc", "tentativeness",
    "self_focus", "social",
    "work", "time_pressure",
    "somatic", "coping"
]


def extract_features(
    text: str,
    retry: int = 2,
    sleep_sec: float = 1.0
) -> dict:
    """
    Extract psycholinguistic features from a single day-level text.

    Parameters
    ----------
    text : str
        Day-level concatenated transcript (Korean).
    retry : int
        Maximum number of retries on malformed JSON.
    sleep_sec : float
        Fixed delay (seconds) after every API call for rate-limit compliance.

    Returns
    -------
    dict
        13 feature scores (float, 0.0-1.0). Returns zeros on repeated failure.
    """
    if text is None or not str(text).strip():
        return {k: 0.0 for k in FEATURE_KEYS}

    for attempt in range(retry + 1):
        try:
            response = client.chat.completions.create(
                model="gpt-5.2-2025-12-11",
                temperature=0.0,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": USER_PROMPT_TEMPLATE.format(text=text)}
                ]
            )
            time.sleep(sleep_sec)

            result = json.loads(response.choices[0].message.content)
            return {k: float(result.get(k, 0.0)) for k in FEATURE_KEYS}

        except Exception as e:
            time.sleep(sleep_sec)
            if attempt == retry:
                print(f"Failed after {retry + 1} attempts: {e}")
                return {k: 0.0 for k in FEATURE_KEYS}

---
## 4. Usage Example

```python
# Single text extraction
text = "Day-level concatenated Korean transcript here..."
scores = extract_features(text)
print(scores)
# {'posemo': 0.2, 'negemo': 0.5, 'anxiety': 0.3, ...}

# Batch extraction over a DataFrame
import pandas as pd

df = pd.read_csv("transcripts.csv")
results = df["text_day"].apply(extract_features).apply(pd.Series)
df = pd.concat([df, results.add_prefix("llm_")], axis=1)
```